In [1]:
import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter
import datetime
import locale
locale.setlocale(locale.LC_TIME, 'ru_RU.UTF-8')
from io import BytesIO

In [2]:
# Словарь месяцев вручную, чтобы избежать проблем с кодировкой
months_ru = {
    1: "янв", 2: "фев", 3: "мар", 4: "апр", 5: "май", 6: "июн",
    7: "июл", 8: "авг", 9: "сен", 10: "окт", 11: "ноя", 12: "дек"
}

In [7]:
df = pd.read_excel(r"C:\Users\m.olshanskiy\Desktop\032026_Продажи_март.xlsx", sheet_name = 'рейтинг')

In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 173 entries, 0 to 172
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   №           173 non-null    int64  
 1   Застройщик  173 non-null    object 
 2   Indicators  154 non-null    float64
dtypes: float64(1), int64(1), object(1)
memory usage: 4.2+ KB


In [19]:
df2

,Застройщик,Регион,Название ЖК,"Среднее за 2024 г., шт./мес",2025-01-01 00:00:00,2025-02-01 00:00:00,2025-03-01 00:00:00,2025-04-01 00:00:00,2025-05-01 00:00:00,2025-06-01 00:00:00,...,2025-08-01 00:00:00,2025-09-01 00:00:00,2025-10-01 00:00:00,2025-11-01 00:00:00,2025-12-01 00:00:00,"Среднее за 2025 г., шт./мес",2026-01-01 00:00:00,2026-02-01 00:00:00,2026-03-01 00:00:00,"Среднее за 2026 г., шт./мес"
0,ПИК,Total,NaN,2016.500000,3034.0,4402.0,3761.0,3091.0,1618.0,1546.0,...,1555.0,1587.0,1777.0,1720.0,1443.0,2282.666667,1237.0,913.0,1173.0,1107.666667
1,ПИК,Москва,Total,1244.666667,1325.0,1893.0,1537.0,1442.0,888.0,939.0,...,934.0,862.0,905.0,969.0,784.0,1130.250000,566.0,438.0,605.0,536.333333
2,ПИК,Москва,Бусиновский парк,112.818182,128.0,208.0,146.0,133.0,104.0,102.0,...,89.0,43.0,43.0,30.0,26.0,94.250000,16.0,21.0,25.0,20.666667
3,ПИК,Москва,Люблинский парк,94.166667,32.0,56.0,59.0,11.0,1.0,94.0,...,103.0,75.0,78.0,56.0,41.0,60.666667,27.0,21.0,9.0,19.000000
4,ПИК,Москва,Зеленый парк,55.750000,142.0,137.0,79.0,95.0,80.0,43.0,...,52.0,68.0,73.0,51.0,57.0,78.333333,37.0,29.0,57.0,41.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1145,ПСТ,Московская область,Три квартала,1.000000,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1146,СК Олимпийский 10,Total,NaN,1.000000,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1147,СК Олимпийский 10,Москва,Total,1.000000,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1148,СК Олимпийский 10,Москва,Соле Хиллс,1.000000,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [10]:
developers = df["Застройщик"].tolist()

In [18]:
df2 = pd.read_excel(r"C:\Users\m.olshanskiy\Desktop\032026_Продажи_март.xlsx", sheet_name = 'массив')

In [20]:
def format_col(col):
    if isinstance(col, (pd.Timestamp, datetime.datetime)):
        return f"{months_ru[col.month]}.{str(col.year)[-2:]}"
    return col

df2.columns = [format_col(col) for col in df2.columns]


In [21]:
# Фильтруем по каждому региону
def remove_total_if_one_jk(group):
    # Проверяем сколько строк с ЖК (не Total)
    jk_count = group[group["Название ЖК"] != "Total"].shape[0]
    if jk_count <= 1:
        # Удаляем строки с Total
        group = group[group["Название ЖК"] != "Total"]
    return group

In [22]:
# Проверяем результат
print(df2.columns.tolist())

['Застройщик', 'Регион', 'Название ЖК', 'Среднее за 2024 г., шт./мес', 'янв.25', 'фев.25', 'мар.25', 'апр.25', 'май.25', 'июн.25', 'июл.25', 'авг.25', 'сен.25', 'окт.25', 'ноя.25', 'дек.25', 'Среднее за 2025 г., шт./мес', 'янв.26', 'фев.26', 'мар.26', 'Среднее за 2026 г., шт./мес']


In [23]:
region_order = ['Total', 'Москва', 'Новая Москва', 'Московская область']

In [24]:
# 2. Создаём словарь: застройщик → порядковый номер
developer_numbers = {name: i + 1 for i, name in enumerate(df["Застройщик"].dropna().unique())}

# 3. Добавляем новый столбец с номерами
df2["№"] = df2["Застройщик"].map(developer_numbers)

# 4. Ставим столбец "№" в начало
df2 = df2[["№"] + [col for col in df2.columns if col != "№"]]





In [25]:
df2['Застройщик'] = pd.Categorical(df2['Застройщик'], categories=developers, ordered=True)
df2['Регион'] = pd.Categorical(df2['Регион'], categories=region_order, ordered=True)
df_sorted = df2.sort_values(['Застройщик', 'Регион']).reset_index(drop=True)

# 5. Дублируем шапку перед каждым застройщиком
header = pd.DataFrame([df_sorted.columns], columns=df_sorted.columns)  # создаём строку-шапку
result = pd.DataFrame(columns=df_sorted.columns)

for dev in df_sorted["№"].unique():
    block = df_sorted[df_sorted["№"] == dev]
    result = pd.concat([result, header, block], ignore_index=True)

df_sorted = result

# 5. Заменяем все значения "Total" на "Итого"
# df_sorted = df_sorted.replace("Total", "Итого")

df_cleaned = df_sorted.groupby("Регион", group_keys=False).apply(remove_total_if_one_jk)

C:\Users\m.olshanskiy\AppData\Local\Temp\ipykernel_20204\2155248353.py:18: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_cleaned = df_sorted.groupby("Регион", group_keys=False).apply(remove_total_if_one_jk)


In [26]:
df_sorted = df_cleaned

In [28]:
import numpy as np

# --- 2. Сохраняем в Excel ---
output_file = r"C:\Users\m.olshanskiy\Desktop\Продажи отсортированные 022026.xlsx"
df_sorted.to_excel(output_file, index=False)

# приводим месяцы к числам
df_sorted['янв.26'] = pd.to_numeric(df_sorted['янв.26'], errors='coerce')
df_sorted['фев.26'] = pd.to_numeric(df_sorted['фев.26'], errors='coerce')

# расчет динамики
df_sorted['Динамика мес/мес,%'] = np.where(
    df_sorted['Регион'].str.strip().str.lower() == 'total',
    (df_sorted['фев.26'] / df_sorted['янв.26'] - 1),
    None
)

# округление до десятых процента
df_sorted['Динамика мес/мес,%'] = df_sorted['Динамика мес/мес,%'].round(3)


# --- 3. Открываем для форматирования ---
wb = load_workbook(output_file)
ws = wb.active

# Эта часть удаляет строки Total там, где в регионе всего один проект

region_start = None
region_end = None
current_region = None
rows_to_delete = []

for row in range(2, ws.max_row + 1):  # первая строка — заголовок
    region = ws[f"C{row}"].value  # теперь регион в колонке C
    jk_name = ws[f"D{row}"].value  # Total / ЖК в колонке D

    if region is None:
        continue
    if jk_name is None:
        jk_name = ""

    if region != current_region:
        if region_start is not None:
            # Собираем все названия ЖК в регионе
            jk_list = [str(ws[f"D{r}"].value).strip() for r in range(region_start, region_end + 1)]
            total_count = sum(1 for name in jk_list if name.lower() == "total")
            real_jk_count = sum(1 for name in jk_list if name.lower() != "total")

            print(f"\nРегион: {current_region}")
            print(f"Список ЖК (включая Total): {jk_list}")
            print(f"Количество ЖК без Total: {real_jk_count}, Total: {total_count}")

            # Если ЖК всего 1, удаляем Total
            if real_jk_count == 1 and total_count > 0:
                for r in range(region_start, region_end + 1):
                    if str(ws[f"D{r}"].value).strip().lower() == "total":
                        rows_to_delete.append(r)
                        print(f"Удаляю строку {r} с 'Total'")

        current_region = region
        region_start = row

    region_end = row

# Проверяем последний регион
if region_start is not None:
    jk_list = [str(ws[f"D{r}"].value).strip() for r in range(region_start, region_end + 1)]
    total_count = sum(1 for name in jk_list if name.lower() == "total")
    real_jk_count = sum(1 for name in jk_list if name.lower() != "total")

    print(f"\nРегион: {current_region}")
    print(f"Список ЖК (включая Total): {jk_list}")
    print(f"Количество ЖК без Total: {real_jk_count}, Total: {total_count}")

    if real_jk_count == 1 and total_count > 0:
        for r in range(region_start, region_end + 1):
            if str(ws[f"D{r}"].value).strip().lower() == "total":
                rows_to_delete.append(r)
                print(f"Удаляю строку {r} с 'Total'")

# Удаляем строки с конца
for r in sorted(rows_to_delete, reverse=True):
    ws.delete_rows(r)

# Эта часть добавляет шапку для каждого застройщика, а также добавляет серую заливку и жирный шрифт

# Форматы
fill = PatternFill(start_color="D9D9D9", end_color="D9D9D9", fill_type="solid")
bold_font = Font(bold=True)
center_align = Alignment(vertical="center", horizontal="center")

# --- 4. Форматируем строки-шапки ---
for row in ws.iter_rows(min_row=1, max_row=ws.max_row):
    if row[0].value == "№":  # шапка начинается со "№"
        for cell in row:
            cell.font = bold_font
            cell.fill = fill
            cell.alignment = center_align

# --- 5. Объединяем одинаковые подряд ячейки ---
def merge_identical_cells(column_idx):
    start = 2  # пропускаем первую строку
    current_value = ws[f"{get_column_letter(column_idx)}{start}"].value
    for row in range(3, ws.max_row + 2):
        cell_value = ws[f"{get_column_letter(column_idx)}{row}"].value
        if cell_value != current_value:
            if row - start > 1 and current_value is not None:
                ws.merge_cells(start_row=start, start_column=column_idx,
                               end_row=row - 1, end_column=column_idx)
                ws[f"{get_column_letter(column_idx)}{start}"].alignment = center_align
            start = row
            current_value = cell_value

# Объединяем по нужным столбцам
merge_identical_cells(1)  # №
merge_identical_cells(2)  # Застройщик
merge_identical_cells(3)  # Регион
merge_identical_cells(4)  # Название ЖК (если нужно)




# --- 6. Сохраняем итог ---
wb.save(output_file)
print("✅ Готово! Шапки добавлены, выделены цветом, и ячейки объединены.")



Регион: Регион
Список ЖК (включая Total): ['Название ЖК']
Количество ЖК без Total: 1, Total: 0

Регион: Total
Список ЖК (включая Total): ['None']
Количество ЖК без Total: 1, Total: 0

Регион: Москва
Список ЖК (включая Total): ['Total', 'Бусиновский парк', 'Люблинский парк', 'Зеленый парк', 'Москворечье', 'Плеханова 11', 'Алтуфьевское 53', 'Матвеевский парк', 'Большая Академическая 85', 'Холланд Парк', 'Волжский парк', 'Кавказский 51', 'Никольские луга', 'Мичуринский парк', 'Второй Иртышский', 'Амурский Парк', 'Руставели 14', 'Полар', 'Митинский лес', 'Второй Нагатинский', 'Первый Дубровский', 'Барклая 6', 'Сигнальный 16', 'Ютаново', 'Кутузовский квартал', 'Кронштадтский 9', 'Квартал Мит', 'Новое Очаково', 'Нарвин', 'Кольская 8', 'Римского-Корсакова 11', 'Онежский вал', 'Открытый парк', 'Лосиноостровский парк', 'Большая Очаковская 2', 'Кронштадтский 14', 'Строгино 360', 'Полярная 25', 'Красноказарменная 15', 'Перовское 2', 'Вангарден']
Количество ЖК без Total: 40, Total: 1

Регион: Нов

In [ ]:
# пока не нужно
# df['Динамика мес/мес,%'] = np.where(
#     df['Регион'] == 'Total',
#     (df['фев.26'] / df['янв.26'] - 1) * 100,
#     None
# )

In [29]:
df_result = pd.read_excel(r"C:\Users\m.olshanskiy\Desktop\Продажи отсортированные 070426.xlsx")

In [32]:
df_result.head()

,№,Застройщик,Регион,Название ЖК,"Среднее за 2024 г., шт./мес",янв.25,фев.25,мар.25,апр.25,май.25,...,авг.25,сен.25,окт.25,ноя.25,дек.25,"Среднее за 2025 г., шт./мес",янв.26,фев.26,мар.26,"Среднее за 2026 г., шт./мес"
0,1,ПИК,Total,NaN,2016.5,3034,4402,3761,3091,1618,...,1555,1587,1777,1720,1443,2282.666667,1237,913,1173,1107.666667
1,1,ПИК,Москва,Total,1244.666667,1325,1893,1537,1442,888,...,934,862,905,969,784,1130.25,566,438,605,536.333333
2,1,ПИК,Москва,Бусиновский парк,112.818182,128,208,146,133,104,...,89,43,43,30,26,94.25,16,21,25,20.666667
3,1,ПИК,Москва,Люблинский парк,94.166667,32,56,59,11,1,...,103,75,78,56,41,60.666667,27,21,9,19
4,1,ПИК,Москва,Зеленый парк,55.75,142,137,79,95,80,...,52,68,73,51,57,78.333333,37,29,57,41


In [31]:
df_result = df_result.ffill()

In [33]:
is_total = df_result['Название ЖК'].astype(str).str.contains('Total', na=False)

In [ ]:
base_df = df_result[~is_total]

grouped = base_df.groupby('Застройщик').agg({
    'Название ЖК': 'nunique',
    'Регион': 'nunique'
}).rename(columns={
    'Название ЖК': 'jk_count',
    'Регион': 'region_count'
})

In [ ]:
df_result = df_result.merge(grouped, on='Застройщик', how='left')

In [ ]:
to_delete = (
    is_total &
    (
        (df_result['jk_count'] == 1) |
        (df_result['region_count'] == 1)
    )
)

In [ ]:
df_result = df_result[~to_delete]

In [ ]:
df_result = df_result.drop(columns=['jk_count', 'region_count'])